In [ ]:
from pathlib import Path
from typing import List, Dict, Tuple, Union
from collections import defaultdict
import json

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm.auto import tqdm
from PIL import Image
import h5py
import scipy.io
from scipy.stats import zscore

import nilearn as nl
import nilearn.image as nl_image
import nilearn.plotting as nl_plotting
import nibabel as nib

In [ ]:
SUBJECTS = [
    f"subj{i:02d}" for i in range(1, 9)
]
SUBJECTS = [SUBJECTS[0]]  # For testing
data_space = "func1pt8mm"
beta_type ="betas_fithrf_GLMdenoise_RR"

# SUBJECTS

In [ ]:
ds_dir = '${MBS_NSD_DIR}'
# ds_path = '${MBS_NSD_DIR}/nsddata_betas/ppdata/subj01/fsaverage/betas_fithrf_GLMdenoise_RR'

ds_dir = Path(ds_dir)

# list((ds_path ).iterdir())

# Noise Ceiling

In [ ]:
ncsnr2nc = lambda x: 100 * (x**2) / (x**2 + 1/3)

In [ ]:
noise_ceiling_masks = {}

for sub in tqdm(SUBJECTS):
    nc_filepath = ds_dir / "nsddata_betas/ppdata" / sub / data_space / beta_type / "ncsnr.nii.gz"
    nc = nib.load(nc_filepath).get_fdata()    
    noise_ceiling_masks[sub] = ncsnr2nc(nc)

In [ ]:
ncsnr2nc(0.2)

In [ ]:
thresh = 10

for sub, mask in noise_ceiling_masks.items():
    print(f"Subject: {sub}, available voxels: {(mask > thresh).sum()}")



# ROI masks

In [ ]:
def load_ctab(file_path):
    ctab = pd.read_csv(
        file_path,
        sep="\s+",   # split on any whitespace
        header=None,
        comment='#',             # just in case there are comments
        # skiprows=1,               # skip the "num entries" line,
        names=['roi_id', 'roi_name']
    )
    return ctab


In [ ]:
ROI_files = [
    "streams", 
    "prf-visualrois",
    "nsdgeneral",
    "floc-words",
    "floc-places",
    "floc-faces",
    "floc-bodies"
]

In [ ]:
ROI_raw_masks = defaultdict(dict)
ROI_metadata = defaultdict(dict)


all_rois = []
for sub in tqdm(SUBJECTS):
    for meta_roi in tqdm(ROI_files, leave=False):
        
        roi_meta_mask = ds_dir / "nsddata/ppdata" / sub / "func1pt8mm/roi" / f"{meta_roi}.nii.gz"

        roi_meta_mask = np.squeeze(nib.load(roi_meta_mask).get_fdata())

        metadata = load_ctab(ds_dir / "nsddata/freesurfer" / sub / "label" / f"{meta_roi}.mgz.ctab")
        ROI_metadata[sub][meta_roi] = metadata
        
        # if sub == "subj01" :
        #     print(meta_roi)
        
        for row_id, row in metadata[1:].iterrows():
            roi_name = row.roi_name
            roi_id = row.roi_id
            
            roi_mask = roi_meta_mask==roi_id
            
            ROI_raw_masks[sub][roi_name] = roi_mask
            all_rois.append(roi_name)
            
            # if sub == "subj01" :
            #     print(f"\t {roi_name}")


    # # Whole brain mask
    # ROI_raw_masks[sub]["whole_brain"] = np.ones_like(roi_meta_mask, dtype=bool)
    # all_rois.append("whole_brain")


all_rois = sorted(set(all_rois))
    


In [ ]:
all_rois

In [ ]:
ROI_mapping = {
   k: k for k in all_rois
}
ROI_version = "individualROIs"

In [ ]:
ROI_mapping = {
    "V1": ["V1d", "V1v"],
    "V2": ["V2d", "V2v"],
    "V3": ["V3d", "V3v"],
    "V4": ["hV4"],
    "IT": [
        "midlateral",
        "midparietal",
        "midventral",
        "parietal",
        "lateral",
        "ventral"
        
    ],
    "VWFA": ["OWFA", "VWFA-1", "VWFA-2", "mfs-words", "mTL-words"],
    "faces": ["OFA", "FFA-1", "FFA-2", "mTL-faces", "aTL-faces"],
    "bodies": ["EBA", "FBA-1", "FBA-2", "mTL-bodies"],
    "places": ["OPA", "PPA", "RSC"],
    "vision": ["nsdgeneral"]
}

ROI_version = "combinedROIs"

In [ ]:
ROI_masks = defaultdict(dict)

for sub in tqdm(SUBJECTS):
    for roi_name, roi_list in ROI_mapping.items():
        roi_mask = np.zeros_like(ROI_raw_masks[sub]["nsdgeneral"], dtype=bool)
        
        for roi_ in roi_list:
        
            roi_mask |= ROI_raw_masks[sub][roi_]
            
        #Apply nsdgeneral mask
        roi_mask &= ROI_raw_masks[sub]["nsdgeneral"]
        
        ROI_masks[sub][roi_name] = roi_mask
        
    # Add whole_brain
    roi_mask = np.ones_like(ROI_raw_masks[sub]["nsdgeneral"], dtype=bool)
    ROI_masks[sub]["whole_brain"] = roi_mask

In [ ]:
for sub in tqdm(SUBJECTS):
    print(sub)
    for roi_name, roi_mask in ROI_masks[sub].items():
        print("\t", roi_name, roi_mask.sum())

# Noise Ceiled ROI Masks

In [ ]:
NOISE_CEILING_THRESHOLD = 10

In [ ]:
ROI_masks_noise_ceiled = defaultdict(dict)

for sub in tqdm(SUBJECTS):
    for roi_name, roi_mask in ROI_masks[sub].items():
        nc_mask = noise_ceiling_masks[sub] >= NOISE_CEILING_THRESHOLD
        ROI_masks_noise_ceiled[sub][roi_name] = roi_mask & nc_mask

In [ ]:
for sub in tqdm(SUBJECTS):
    print(sub)
    for roi_name, roi_mask in ROI_masks_noise_ceiled[sub].items():
        print("\t", roi_name, roi_mask.sum())

In [ ]:
# break

# Process data

In [ ]:
subject_data = defaultdict(dict)

for sub in tqdm(SUBJECTS):
    sub_data_dir = ds_dir / "nsddata_betas/ppdata" / sub / data_space / beta_type
    
    subj_data = []
    func_data_paths = sorted(list(sub_data_dir.glob("betas_session*.hdf5")))

    for path in tqdm(func_data_paths, total=len(func_data_paths), leave=False):
        with h5py.File(path, 'r') as func_data_file:
            print(func_data_file['betas'].shape)
            func_data = func_data_file['betas'][:]  # 750 x Nz x Ny x Nx
            func_data = func_data.transpose((0, 3, 2, 1))  # 750 x Nx x Ny x Nz
        # func_data = np.transpose(func_data, (0, 3, 2, 1))
        func_data = func_data.astype(np.float16) / 300
        func_data = zscore(func_data, axis=0, ddof=1)
        subj_data.append(func_data)
        
        # break

    subject_data[sub] = np.concatenate(subj_data, axis=0) # (750 x n_sessions) x Nx x Ny x Nz


In [ ]:
# subject_data = defaultdict(dict)

# for sub in tqdm(SUBJECTS):
#     sub_data_dir = ds_dir / "nsddata_betas/ppdata" / sub / data_space / beta_type
    
#     subj_data = []
#     func_data_paths = sorted(list(sub_data_dir.glob("betas_session*.nii.gz")))

#     for path in tqdm(func_data_paths, total=len(func_data_paths), leave=False):
#         func_data = nib.load(path).get_fdata()  # 750 x Nx x Ny x Nz
#         print(func_data.shape)
#         # func_data = np.transpose(func_data, (0, 3, 2, 1))
#         func_data = func_data.astype(np.float16) / 300
#         func_data = zscore(func_data, axis=0, ddof=1)
#         subj_data.append(func_data)
        
#         # break

#     subject_data[sub] = np.concatenate(subj_data, axis=0) # (750 x n_sessions) x Nx x Ny x Nz

In [ ]:
def split_stimuli_by_reps(df_stimuli: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split stimulus presentations into two DataFrames based on repetition count per nsdId.

    Parameters:
        df_stimuli (pd.DataFrame): Long-form DataFrame of presentations with at least:
            - 'nsdId' (int): stimulus identifier
            - 'rep' (int): repetition index for that presentation

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]:
            - stimuli_with_all_reps: subset of rows whose nsdId appears exactly 3 times
            - stimuli_fewer_reps: subset of rows whose nsdId appears fewer than 3 times

    Notes:
        - Assumes up to 3 planned repetitions per stimulus and counts rows per 'nsdId'.
        - The returned DataFrames preserve the original columns and row order of the input.
        - To get unique stimulus IDs per set, use `.nsdId.unique()` on the returned DataFrames.
    """
    stimuli_with_all_reps = []
    stimuli_missing_reps = []
    for group, group_data in df_stimuli.groupby('nsdId'):
        if group_data.shape[0] == 3:
            stimuli_with_all_reps.append(group)
        else:
            stimuli_missing_reps.append(group)
    stimuli_with_all_reps = df_stimuli[df_stimuli.nsdId.isin(stimuli_with_all_reps)]
    stimuli_missing_reps = df_stimuli[df_stimuli.nsdId.isin(stimuli_missing_reps)]
    return stimuli_with_all_reps, stimuli_missing_reps


def process_trials_with_missing_reps(
    stimuli_missing_reps: pd.DataFrame,
    subject_brain_data: np.ndarray
) -> List[np.ndarray]:
    """
    Process trials with missing repetitions by averaging available brain data per stimulus.

    Parameters:
        stimuli_missing_reps (pd.DataFrame): DataFrame of presentations with missing reps, must include:
            - 'nsdId' (int): stimulus identifier
            - 'trial' (int): trial index corresponding to brain data columns
        subject_brain_data (np.ndarray): Brain data array of shape (n_voxels, n_trials)

    Returns:
        np.ndarray: Array of shape (n_voxels, n_stimuli) containing averaged brain data for each stimulus.
    """
    data_missing_reps = []
    for nsdId, group_data in tqdm(stimuli_missing_reps.groupby('nsdId'), leave=False, desc=""):
        brain_data_stimulus = subject_brain_data[..., group_data.index.values].mean(axis=-1)
        data_missing_reps.append(brain_data_stimulus)
    
    return np.stack(data_missing_reps, axis=-1)


def process_sub_data_split(subject_brain_data: np.ndarray, df_stimuli: pd.DataFrame, keep_reps: bool = False) -> np.ndarray:
    """
    Process subject brain data by splitting stimuli into those with all repetitions and those with missing reps.

    Parameters:
        subject_brain_data (np.ndarray): Brain data array of shape (n_voxels, n_trials)
        df_stimuli (pd.DataFrame): DataFrame of presentations with at least:
            - 'nsdId' (int): stimulus identifier
            - 'trial' (int): trial index corresponding to brain data columns
            - 'rep' (int): repetition index for that presentation
        keep_reps (bool, optional): If True, do not average repetitions for stimuli with all reps. Defaults to False.

    Returns:
        np.ndarray: Combined brain data array of shape (n_voxels, n_stimuli) for all stimuli.
    """
    stimuli_with_all_reps, stimuli_missing_reps = split_stimuli_by_reps(df_stimuli)
    
    assert keep_reps == False or len(stimuli_missing_reps) == 0, "Cannot keep repetitions when there are stimuli with missing repetitions."

    # Process stimuli with all repetitions
    subject_data_all_reps = subject_brain_data[:, stimuli_with_all_reps.index.values]
    subject_data_all_reps = subject_data_all_reps.reshape(
        subject_data_all_reps.shape[0],
        -1, 3
    )
    if not keep_reps:
        subject_data_all_reps = subject_data_all_reps.mean(axis=2)

    # Process stimuli with missing repetitions
    if len(stimuli_missing_reps) > 0:
        subject_data_missing_reps = process_trials_with_missing_reps(
            stimuli_missing_reps,
            subject_brain_data
        )
    else:
        if keep_reps:
            subject_data_missing_reps = np.empty((*subject_data_all_reps.shape[:-2], 0, subject_data_all_reps.shape[-1]))
        else:
            subject_data_missing_reps = np.empty((*subject_data_all_reps.shape[-1], 0))

    # Combine both sets of processed data
    combined_data = np.concatenate([subject_data_all_reps, subject_data_missing_reps], axis=1)
    return combined_data

def get_subject_stimuli(df_stimuli: pd.DataFrame, subject_id: int = 1) -> pd.DataFrame:
    """
    Return a copy of rows for a single subject.

    Parameters:
        df_stimuli (pd.DataFrame): Long-form DataFrame containing stimulus presentations.
            Must include column:
            - 'subject' (int): subject identifier (1–8 for NSD).
        subject_id (int, optional): Subject to select. Defaults to 1.

    Returns:
        pd.DataFrame: Copy of df_stimuli filtered to the given subject.
    """
    subject_stimuli = df_stimuli[df_stimuli.subject == subject_id].copy()
    return subject_stimuli


def get_valid_stimuli(
    subject_brain_data: np.ndarray,
    df_stimuli: pd.DataFrame
) -> pd.DataFrame:
    """
    Filter presentations to those with a trial index within the completed scans.

    Parameters:
        subject_brain_data (np.ndarray): Brain data array of shape (n_voxels, n_trials).
        df_stimuli (pd.DataFrame): Presentations DataFrame with at least:
            - 'trial' (int): 1-based trial index aligned to brain data columns.
            - 'nsdId' (int): stimulus identifier.
            - 'rep' (int): repetition index.

    Returns:
        pd.DataFrame: Copy of valid rows (trial <= completed n_trials), index reset and
        sorted by ['nsdId', 'rep'].
    """
    completed_trials = subject_brain_data.shape[1]

    valid_stimuli = df_stimuli[df_stimuli.trial <= completed_trials].copy()
    valid_stimuli.reset_index(drop=True, inplace=True)
    valid_stimuli.sort_values(['nsdId', 'rep'], inplace=True)
    return valid_stimuli

def get_stimuli_train_test_splits(df_stimuli: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split presentations into train/test by the NSD shared1000 set.

    Parameters:
        df_stimuli (pd.DataFrame): Presentations DataFrame with:
            - 'shared1000' (bool): True if the stimulus belongs to the shared1000 test set.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]:
            - sub_stim_train: copy of rows with shared1000 == False
            - sub_stim_test: copy of rows with shared1000 == True
    """
    test_mask = df_stimuli["shared1000"] == True
    sub_stim_test = df_stimuli[test_mask].copy()
    sub_stim_train = df_stimuli[~test_mask].copy()
    return sub_stim_train, sub_stim_test


def process_subject_data(
    subject_brain_data: np.ndarray,
    df_stimuli: pd.DataFrame,
    keep_reps: bool = False
) -> Dict[str, np.ndarray]:
    
    df_stimuli = get_valid_stimuli(subject_brain_data, df_stimuli)
    
    df_stimuli_train, df_stimuli_test = get_stimuli_train_test_splits(df_stimuli)
    
    subject_data_train = process_sub_data_split(subject_brain_data, df_stimuli_train, keep_reps=keep_reps)
    subject_data_test = process_sub_data_split(subject_brain_data, df_stimuli_test, keep_reps=keep_reps)
    
    return subject_data_train, subject_data_test, df_stimuli_train, df_stimuli_test
    
    

In [ ]:
save_path = "${MBS_DATA_PREP_OUTPUT_DIR}/nsd_stim_mapping.csv"
df_stim_all = pd.read_csv(save_path)

In [ ]:
raise ValueError("Stop here for testing")

In [ ]:
sub_data_train = {}
sub_data_test = {}

for sub in tqdm(SUBJECTS):
    sub_id = int(sub.replace("subj", ""))
    sub_brain_data = subject_data[sub]
    sub_stim_data = get_subject_stimuli(df_stim_all, sub_id)

    subject_data_train, subject_data_test, df_stimuli_train, df_stimuli_test = process_subject_data(sub_brain_data, sub_stim_data, keep_reps=True)

In [ ]:
raise ValueError("Stop here for testing")

In [ ]:
# func_data_paths

# Save

In [ ]:
save_dir = "${MBS_DATA_PREP_OUTPUT_DIR}"
save_dir = Path(save_dir)

save_dir = save_dir / f"nsd_{data_space}_{ROI_version}.hdf5"

if not save_dir.parent.exists():
    save_dir.parent.mkdir(parents=True, exist_ok=False)

In [ ]:
with h5py.File(save_dir, 'w') as f:
    f.attrs['subjects'] = SUBJECTS
    f.attrs['data_space'] = data_space
    f.attrs['beta_type'] = beta_type
    f.attrs['rois'] = list(ROI_mapping.keys())
    f.attrs['ROI_mapping'] = json.dumps(ROI_mapping)
    
    # Create noise ceiling dataset
    for sub in tqdm(SUBJECTS, desc="Creating noise ceiling masks"):
        f.create_dataset(f"noise_ceiling_masks/{sub}", data=subject_data[sub])
        
    # Create ROI masks
    for sub in tqdm(SUBJECTS, desc="Creating ROI masks"):
        for roi_name, roi_mask in ROI_masks[sub].items():
            f.create_dataset(f"roi_masks/{sub}/{roi_name}", data=roi_mask)
            
    # Create noise ceiled ROI masks
    for sub in tqdm(SUBJECTS, desc="Creating noise ceiled ROI masks"):
        for roi_name, roi_mask in ROI_masks_noise_ceiled[sub].items():
            f.create_dataset(f"roi_masks_noise_ceiling/{sub}/{roi_name}", data=roi_mask)

    # Write data
    for sub in tqdm(SUBJECTS, desc="Writing data"):
        for roi_name, roi_mask in ROI_masks_noise_ceiled[sub].items():
            if roi_name=="whole_brain":continue
            f.create_dataset(f"data/{sub}/{roi_name}", data=subject_data[sub][:, roi_mask])

In [ ]:
subject_data[sub].shape

In [ ]:
roi_mask.shape

# Test

In [ ]:
data_path = "${MBS_DATA_PREP_OUTPUT_DIR}"
data_path = Path(data_path)
data_path = data_path / f"nsd_{data_space}.hdf5"
assert data_path.exists(), "Data path does not exist"

In [ ]:
with h5py.File(data_path, 'r') as f:
    subjects = f.attrs['subjects']
    rois = f.attrs['rois']
    
    for sub in tqdm(subjects):
        print(f"Subject: {sub}")
        for roi in rois:
            print(f"\t ROI: {roi}", f[f"data/{sub}/{roi}"].shape)
    